# 第 3 章 重回帰: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/econometrics-programming/03-multiple-regression.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
seed = 0             # 乱数の種
n = 50               # 標本の大きさ
correlation = 0.9    # x1 と x2 の相関（強い場合）
repetitions = 1000   # 標本を取り直す回数

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
sys.path[:] = [p for p in sys.path if p != str(_root)]
sys.path.insert(0, str(_root))
_lib = sys.modules.get("lib")
if _lib is not None and Path(getattr(_lib, "__file__", "") or "").resolve() != (_root / "lib" / "__init__.py").resolve():
    for _name in list(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 説明変数の相関と、係数のばらつき

In [ ]:
import numpy as np

from lib.chapter03.multiple import multicollinearity_experiment

low = multicollinearity_experiment(0.0, n, repetitions, np.random.default_rng(seed))
high = multicollinearity_experiment(
    correlation, n, repetitions, np.random.default_rng(seed)
)
print(f"相関 0.0: x1 の係数の平均 {low['x1'].mean():.3f}、"
      f"標準偏差 {low['x1'].std(ddof=1):.3f}")
print(f"相関 {correlation}: x1 の係数の平均 {high['x1'].mean():.3f}、"
      f"標準偏差 {high['x1'].std(ddof=1):.3f}")

## 図: 係数の推定値の分布

In [ ]:
from lib.chapter03.plot import plot_collinearity_distributions

plot_collinearity_distributions(low["x1"].tolist(), high["x1"].tolist())

## 試してみよう

- `correlation` を 0.99 にすると、x1 の係数のばらつきは 0.148 から 1.003 へ、約 7
  倍に広がります。平均は 0.991 で、真の値（1）のままです（多重共線性は、偏りでなく、
  ばらつきを広げます）
- `n` を 200 にすると、相関 0.9 でも、ばらつきは 0.162 まで下がります（相関 0.0 は
  0.070）。標本を大きくすると、共線性の影響は小さくなります